In [21]:
## importing libraries
import pandas as pd
from tqdm.auto import tqdm
import numpy as np 
import re
import matplotlib.pyplot as plt

from transformers import BertModel, BertTokenizer
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F
from torch import nn

In [22]:
df_1= pd.read_csv("twitter_data/tweets_devset.txt", sep='\t') 
df_2= pd.read_csv("twitter_data/tweets_testset.txt", sep='\t')

# Write the DataFrame to a CSV file
df_1.to_csv('twitter_data/tweets_devset.csv', index=False)
df_2.to_csv('twitter_data/tweets_testset.csv', index=False)

In [23]:
df_1=pd.read_csv("twitter_data/tweets_devset.csv")
df_2=pd.read_csv("twitter_data/tweets_testset.csv")

In [24]:
df_1.shape,df_2.shape

((14277, 7), (3755, 7))

In [25]:
df_1.head()

,tweetId,tweetText,userId,imageId(s),username,timestamp,label
0,263046056240115712,¿Se acuerdan de la película: “El día después d...,21226711,sandyA_fake_46,iAnnieM,Mon Oct 29 22:34:01 +0000 2012,fake
1,262995061304852481,@milenagimon: Miren a Sandy en NY! Tremenda i...,192378571,sandyA_fake_09,CarlosVerareal,Mon Oct 29 19:11:23 +0000 2012,fake
2,262979898002534400,"Buena la foto del Huracán Sandy, me recuerda a...",132303095,sandyA_fake_09,LucasPalape,Mon Oct 29 18:11:08 +0000 2012,fake
3,262996108400271360,Scary shit #hurricane #NY http://t.co/e4JLBUfH,241995902,sandyA_fake_29,Haaaaarryyy,Mon Oct 29 19:15:33 +0000 2012,fake
4,263018881839411200,My fave place in the world #nyc #hurricane #sa...,250315890,sandyA_fake_15,princess__natt,Mon Oct 29 20:46:02 +0000 2012,fake


In [26]:
import os
def build_image_map(image_root):

    image_map = {}

    for root, dirs, files in os.walk(image_root):
        for file in files:
            if file.endswith(".jpg") or file.endswith(".png"):
                image_id = file.split(".")[0]
                image_map[image_id] = os.path.join(root, file)

    return image_map

In [27]:
image_root = "twitter_data\MediaEval2015_DevSet_Images\Medieval2015_DevSet_Images"

image_map = build_image_map(image_root)

df_1 = df_1[df_1["imageId(s)"].isin(image_map.keys())].reset_index(drop=True)
df_1.shape

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Anirban Banerjee\AppData\Local\Temp\ipykernel_5516\3643827139.py:1: SyntaxWarning: invalid escape sequence '\M'
  image_root = "twitter_data\MediaEval2015_DevSet_Images\Medieval2015_DevSet_Images"


(14092, 7)

In [28]:
def text_preprocessing(text):
    """
    - Remove entity mentions (eg. '@united')
    - Correct errors (eg. '&amp;' to '&')
    @param    text (str): a string to be processed.
    @return   text (Str): the processed string.
    """
    # Remove '@name'
    text = re.sub(r'(@.*?)[\s]', ' ', text)

    # Replace '&amp;' with '&'
    text = re.sub(r'&amp;', '&', text)

    # Remove trailing whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    #removes links
    text = re.sub(r'(?P<url>https?://[^\s]+)', r'', text)
    
    # remove @usernames
    text = re.sub(r"\@(\w+)", "", text)
    
    # remove hashtags
    text = text.replace('#','')
    

    return text

In [29]:
df_1["tweetText"] = df_1["tweetText"].apply(text_preprocessing)

In [30]:
df_1["tweetText"][86]

'newyork weather hurricane sandy god bless audhubillah storm 🌊☔⚡ '

In [31]:
class WeightedBERT(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        
        self.bert = BertModel.from_pretrained(
            'bert-base-uncased',
            output_hidden_states=True
        )
        
        # Learnable weights for 4 layers
        self.layer_weights = nn.Parameter(torch.ones(4))
        
        # Final projection
        self.linear = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, input_ids, attention_mask, return_tokens=False):
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask)
        
        hidden_states = outputs.hidden_states
        
        layers=hidden_states[-4:] #get the last 4 layers
        
        layers = torch.stack(layers, dim=0)  # [B, 4, 768]
        
        # Softmax weights
        alpha = torch.softmax(self.layer_weights, dim=0)
        
        # Weighted sum
        weighted_sum = (alpha[:,None, None, None] * layers).sum(dim=0)
        
        cls_embedding = weighted_sum[:, 0, :]
        token_embeddings = weighted_sum

        cls_embedding=self.linear(cls_embedding)

        if return_tokens:
            return cls_embedding,token_embeddings
        return cls_embedding

In [32]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [33]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = WeightedBERT().to(device)
model.eval()

WeightedBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

In [34]:
model=model.to(device)

In [35]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [36]:
text_feature_dict = {}

for idx, row in df_1.iterrows():
    
    tweet_id = row["tweetId"]
    text = row["tweetText"]
    
    # -------------------------
    # BERT ENCODING
    # -------------------------
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        global_feat, token_emb = model(
            inputs["input_ids"],
            inputs["attention_mask"],
            return_tokens=True
        )

    token_emb = token_emb.squeeze(0).cpu()

    bert_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # -------------------------
    # SPACY PARSING
    # -------------------------
    doc = nlp(text)
    
    nodes=[]
    node_features=[]
    spacy_to_feature_idx = {}

    important_pos = ["NOUN", "PROPN", "VERB", "ADJ"]

    for token in doc:
        # Define nodes (important words)
        if token.pos_ in important_pos:
            word=token.text.lower()

            # find matching indices in bert_tokens
            indices = [
                i for i, t in enumerate(bert_tokens)
                if word in t.replace("##", "")
            ]

            if len(indices) > 0:
                emb = token_emb[indices].mean(dim=0)
                spacy_to_feature_idx[token.i] = len(node_features)
                nodes.append({
                    "text": word,
                    "pos": token.pos_,
                    "index": token.i
                })
                node_features.append(emb)

    # Dependency edges
    edges = []
    for token in doc:
        if token.head.i in spacy_to_feature_idx and token.i in spacy_to_feature_idx:
            if token.head != token:
                edges.append({
                    "from": spacy_to_feature_idx[token.head.i],
                    "to": spacy_to_feature_idx[token.i],
                    "relation": token.dep_
                })
            
    if node_features:  # Only add to dict if we have nodes

        x_tensor = torch.stack(node_features)

        edge_index = torch.tensor([
            [e["from"] for e in edges],
            [e["to"] for e in edges]
        ], dtype=torch.long)
        
        text_feature_dict[tweet_id] = {
            "global": global_feat.squeeze(0).cpu(),
            "x": x_tensor,
            "edge_index": edge_index,
            "nodes": nodes,
            "edges": edges
        }
    

In [38]:
text_feature_dict[df_1["tweetId"].iloc[89]]["x"].shape

torch.Size([6, 768])

In [39]:
from torch_geometric.nn import GATConv
import torch.nn.functional as F

class TextGAT(nn.Module):
    def __init__(self):
        super().__init__()

        self.gat1=GATConv(
            in_channels=768,
            out_channels=256,
            heads=4,
            concat=True
        )

        self.gat2=GATConv(
            in_channels=256*4,
            out_channels=256,
            heads=1,
            concat=False
        )
    def forward(self,x,edge_index):
        x=self.gat1(x,edge_index)
        x=F.elu(x)

        x=self.gat2(x,edge_index)

        return x #[num_nodes,256]

In [43]:
x=x_tensor.to(torch.float32)

In [44]:
gat_model=TextGAT().to(device)

x=x.to(device)
edge_index=edge_index.to(device)
text_enriched_ftrs=gat_model(x,edge_index)